# day three

## RAG 检索-增强 生成
## Retrieval-Augmented Generation
## 查询-检索-增强-生成-答案

In [2]:
# 无RAG模式
from langchain.agents import create_agent
from dotenv import load_dotenv

load_dotenv()

agent = create_agent(
    model="deepseek:deepseek-chat",
)

results = agent.invoke(
    {"messages": [{"role": "human", "content": "讲一下3i/Atlas."}]}
)

In [3]:
print(results["messages"][-1].content)

好的，我们来详细讲解一下 **3i / Atlas**。

这是一个在机器人、人工智能和自动驾驶领域非常重要的概念。它通常指的是 **波士顿动力公司（Boston Dynamics）** 开发的 **足式机器人软件系统**。

我们可以从几个层面来理解它：

### 1. 核心含义：3i 与 Atlas 的关系

*   **Atlas（阿特拉斯）**：这是波士顿动力最著名的**人形机器人硬件平台**。它以其惊人的动态平衡能力、跑酷、后空翻等敏捷动作而闻名于世。
*   **3i**：这并不是一个独立的机器人，而是指驱动Atlas（以及波士顿动力其他机器人，如Spot）的**核心软件架构、控制算法和人工智能系统**。你可以把它理解为机器人的“大脑”和“小脑”。

**简单比喻：**
*   **Atlas** = 顶尖运动员的身体（骨骼、肌肉、传感器）。
*   **3i** = 这位运动员的大脑（进行感知、规划、决策） + 极其强大的小脑（负责实时平衡、协调和反射控制）。

所以，当人们说“3i/Atlas”时，通常指的是 **“搭载了3i系统的Atlas机器人”** 这一整体。

---

### 2. 3i 系统的核心组成部分与原理

3i系统是一个复杂的集成体，其名称可能源于 **“集成”、“智能”、“创新”** 等理念。它的核心技术可以分解为：

**a. 感知与理解（Perception）**
*   Atlas通过头部的多目视觉传感器、激光雷达（LIDAR）和身体的惯性测量单元（IMU）等来感知世界。
*   3i系统会实时处理这些数据，构建机器人周围环境的**3D地图**，并识别物体（如木板、障碍物、手柄）、判断地形特征和计算落脚点。

**b. 模型预测控制与实时规划（MPC & Real-time Planning）**
*   这是3i的“魔法”所在。系统内部有一个精确的**机器人动力学物理模型**。
*   在每毫秒级别，系统都会：
    1.  **预测**：根据当前状态（姿势、速度）和可能的动作，预测未来短时间内机器人的状态。
    2.  **优化**：从成千上万种可能的动作轨迹中，快速计算出一条能完成目标（如跳上高台）且**最稳定、最节能**的轨迹。
    3.  **调整**：根据最新的传感器反馈，不断微调和优化这条轨迹，以应

## 构建数据向量库（网页）

In [4]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [5]:
import os
import shutil

if os.path.exists("./day_three_chroma_rag_db"):
    shutil.rmtree("./day_three_chroma_rag_db")

In [6]:
# 1.读取网页  按照页管理  Document  list[Document]
page_urls = [
    "https://news.sciencenet.cn/htmlnews/2025/7/547139.shtm",
    "http://mrdx.cn/content/20250709/Articel05005NU.htm",
    "https://starwalk.space/zh-Hant/news/3i-atlas-interstellar-object",
    "https://news.sjtu.edu.cn/jdzh/20251205/217724.html"
]

In [8]:
import bs4

bs4_strainer = bs4.SoupStrainer()
loader = WebBaseLoader(
    web_paths=page_urls,
    bs_kwargs={"parse_only": bs4_strainer},
)

docs = loader.load()
print(docs)


[Document(metadata={'source': 'https://news.sciencenet.cn/htmlnews/2025/7/547139.shtm', 'title': '天文学家确认发现第三颗星际天体—新闻—科学网', 'language': 'No language found.'}, page_content='\n\n天文学家确认发现第三颗星际天体—新闻—科学网\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\r\n           \ufeff\n\n\n\n\n生命科学 | \r\n                        医学科学 | \r\n                        化学科学 | \r\n                        工程材料 | \r\n                        信息科学 | \r\n                        地球科学 | \r\n                        数理科学 | \r\n                        管理综合 \n 站内规定 | 手机版\n\n首页 | 新闻 | 博客 | 院士 | 人才 | 会议 | 基金·项目 | 论文 | 视频·直播 | 小柯机器人 | 医学科普\n\n\n\n\n\n\n\n\n\n\n\n\n\n\r\n            \xa0\n\n\n\n\n\n\n\n\n\r\n                                            作者：张佳欣 来源：?科技日报 发布时间：2025/7/4 14:52:54\n\r\n                                            选择字号：小 中 \r\n                                                        大\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\r\n                                                        天文学家确认发现第三颗星际天体\n\n\n\n\n\n\n\xa0它不是鸟，不是飞机，也不

In [9]:
# 2.分割文本  文本段（chunk）  Document  list[Document]
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=100,
    add_start_index=True
)
all_splits = text_splitter.split_documents(docs)
print(all_splits)

[Document(metadata={'source': 'https://news.sciencenet.cn/htmlnews/2025/7/547139.shtm', 'title': '天文学家确认发现第三颗星际天体—新闻—科学网', 'language': 'No language found.', 'start_index': 2}, page_content='天文学家确认发现第三颗星际天体—新闻—科学网\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\r\n           \ufeff\n\n\n\n\n生命科学 | \r\n                        医学科学 | \r\n                        化学科学 | \r\n                        工程材料 | \r\n                        信息科学 | \r\n                        地球科学 | \r\n                        数理科学 | \r\n                        管理综合 \n 站内规定 | 手机版\n\n首页 | 新闻 | 博客 | 院士 | 人才 | 会议 | 基金·项目 | 论文 | 视频·直播 | 小柯机器人 | 医学科普\n\n\n\n\n\n\n\n\n\n\n\n\n\n\r\n            \xa0\n\n\n\n\n\n\n\n\n\r\n                                            作者：张佳欣 来源：?科技日报 发布时间：2025/7/4 14:52:54\n\r\n                                            选择字号：小 中 \r\n                                                        大\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\r\n                                                        天文学家确认发现第三颗星际天体\n\n\n\n\n\n\n\x

In [10]:
# 3.嵌入模型
embedding = OllamaEmbeddings(
    model="qwen3-embedding:8b"
)

In [11]:
# 4 向量库：把多个文本段/向量存入向量库
vectory_store = Chroma(
    collection_name="rag_collection",
    embedding_function=embedding,
    persist_directory="./day_three_chroma_rag_db",
)
ids = vectory_store.add_documents(documents=all_splits)
print(ids)

['3fa4fa36-5cb6-46b5-b437-fcfad514412b', '0a48559e-7941-492a-a837-8038a006df52', 'd08aa1e1-f682-4a92-9ca8-2bcce726a85b', 'eebc1a24-59d2-4137-ad7a-10bca961c39b', '9a303bdc-09a8-4bb9-b252-8f50ee82c163', 'ca05f592-ac49-4638-87b0-b23ae7eb5a6d', '9f02ed4b-de4c-420c-b493-1c4965531c02', 'fe2a5e02-09ea-4f95-b55c-5f6b030d7c1b', 'a1ebd6b9-8a69-4c12-a7fa-96d3fe6bdfcb', '1272c225-86b8-408b-abc4-319c326630bd', 'e03f12e5-6fd0-4273-b0ad-6e6037923bba', 'aa58e48c-af41-4208-b719-b2ca5f3ac8ac', '30841425-46c6-4652-95cb-836cfde0aaac', 'd8979400-8507-4ca9-86aa-07d0dd2c9417', '99375134-40cd-4d4a-8b7f-96c24ad57850', '6f6ba212-07d6-4ab6-9876-10c8507fb695', '309988b7-613b-41bc-a87b-8a6ba2177fee', 'b45d8c27-0555-45cb-9c99-af5036263fc6']


## 简单RAG案例

In [12]:
from langchain.agents import create_agent
from dotenv import load_dotenv
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain.tools import tool

In [14]:
# 创建检索工具
@tool(response_format="content_and_artifact")
def retrieve_content(query: str):
    """
    检索数据库的工具，来帮助回答用户的问题
    :param query: str
    :return: content_and_artifact
    """
    retrieve_docs = vectory_store.similarity_search(query, k=1)
    content = "\n\n".join(
        f"Source:{doc.metadata}\nContent:{doc.page_content}" for doc in retrieve_docs
    )
    return content, retrieve_docs

In [13]:
# 连接向量数据库
embedding = OllamaEmbeddings(
    model="qwen3-embedding:8b"
)
vectory_store = Chroma(
    collection_name="rag_collection",
    embedding_function=embedding, \
    persist_directory="./day_three_chroma_rag_db",
)

In [15]:
load_dotenv()
system_prompt = """你是一个天体爱好者，可以通过检索工具的辅助回答用户问题。"""
agent = create_agent(
    model="deepseek:deepseek-chat",
    system_prompt=system_prompt,
    tools=[retrieve_content],
)
results = agent.invoke(
    {"messages": [{"role": "human", "content": "讲一下3i/Atlas."}]}
)

In [16]:
messages = results["messages"]
print(f"历史消息：{len(messages)}条")
for message in messages:
    message.pretty_print()

历史消息：8条
================================ Human Message =================================

讲一下3i/Atlas.
================================== Ai Message ==================================

我来帮您查找关于3i/Atlas的信息。
Tool Calls:
  retrieve_content (call_00_O4NEdyUuJ2X7qGKPcnvoWp6t)
 Call ID: call_00_O4NEdyUuJ2X7qGKPcnvoWp6t
  Args:
    query: 3i/Atlas 彗星 天体
================================= Tool Message =================================
Name: retrieve_content

Source:{'source': 'https://starwalk.space/zh-Hant/news/3i-atlas-interstellar-object', 'description': '2025年12月19日，罕見的星際彗星3I/ATLAS將迎來其與地球的最近距離！這有多危險？了解它對我們星球意味著什麽。\n', 'start_index': 1194, 'title': '3I ATLAS最新消息 | 3I ATLAS目前位置 | 3I ATLAS外星人, 最新照片 | 3I/亞特拉斯彗星 | Star Walk', 'language': 'zh-Hant'}
Content:什麽是3I/ATLAS？3I/ATLAS 是已知的第三顆 星際天體——來自太陽系之外的稀有訪客。它于2025年7月1日首次被智利的ATLAS巡天望遠鏡發現。官方觀點（由NASA、ESA以及大多數天文學家支持）很明確：3I/ATLAS是一顆天然彗星——這是繼‘奧陌陌’和彗星2I/鮑裏索夫之後確認的第三個星際天體。但並非所有人都信服，一些人認為它的不尋常特征為更具異域色彩的解釋留下了空間。3I/ATLAS 是外星飛船還是彗星？哈佛教授 vs 科學界自從被發現以來，哈佛天文學家 Avi 

In [17]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

In [22]:
checkpointer = InMemorySaver()

agent = create_agent(
    model="deepseek:deepseek-chat",
    system_prompt=system_prompt,
    tools=[retrieve_content],
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            "retrieve_content": True
        }
    )],
    checkpointer=checkpointer,
)

In [23]:
config = {"configurable": {"thread_id": "2"}}
results = agent.invoke(
    {"messages": [{"role": "human", "content": "讲一下3i/Atlas."}]},
    config=config,
)
messages = results["messages"]
print(f"历史消息：{len(messages)}条")
for message in messages:
    message.pretty_print()

if "__interrupt__" in results:
    print("INTERRUPTED")
    interrupt = results["__interrupt__"][0]
    for request in interrupt.value["action_requests"]:
        print(request["description"])

results = agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config,
)

历史消息：2条
================================ Human Message =================================

讲一下3i/Atlas.
================================== Ai Message ==================================

我来帮您查询关于3i/Atlas的信息。
Tool Calls:
  retrieve_content (call_00_JY1oCOy5AwGCGaEgAK8oWg5g)
 Call ID: call_00_JY1oCOy5AwGCGaEgAK8oWg5g
  Args:
    query: 3i/Atlas 彗星 天体
INTERRUPTED
Tool execution requires approval

Tool: retrieve_content
Args: {'query': '3i/Atlas 彗星 天体'}


In [24]:
messages = results["messages"]
print(f"历史消息：{len(messages)}条")
for message in messages:
    message.pretty_print()

历史消息：4条
================================ Human Message =================================

讲一下3i/Atlas.
================================== Ai Message ==================================

我来帮您查询关于3i/Atlas的信息。
Tool Calls:
  retrieve_content (call_00_JY1oCOy5AwGCGaEgAK8oWg5g)
 Call ID: call_00_JY1oCOy5AwGCGaEgAK8oWg5g
  Args:
    query: 3i/Atlas 彗星 天体
================================= Tool Message =================================
Name: retrieve_content

Source:{'title': '3I ATLAS最新消息 | 3I ATLAS目前位置 | 3I ATLAS外星人, 最新照片 | 3I/亞特拉斯彗星 | Star Walk', 'language': 'zh-Hant', 'description': '2025年12月19日，罕見的星際彗星3I/ATLAS將迎來其與地球的最近距離！這有多危險？了解它對我們星球意味著什麽。\n', 'source': 'https://starwalk.space/zh-Hant/news/3i-atlas-interstellar-object', 'start_index': 1194}
Content:什麽是3I/ATLAS？3I/ATLAS 是已知的第三顆 星際天體——來自太陽系之外的稀有訪客。它于2025年7月1日首次被智利的ATLAS巡天望遠鏡發現。官方觀點（由NASA、ESA以及大多數天文學家支持）很明確：3I/ATLAS是一顆天然彗星——這是繼‘奧陌陌’和彗星2I/鮑裏索夫之後確認的第三個星際天體。但並非所有人都信服，一些人認為它的不尋常特征為更具異域色彩的解釋留下了空間。3I/ATLAS 是外星飛船還是彗星？哈佛教授 vs 科學界自從被發現以來，哈佛天文學家 Avi 

In [25]:
if "__interrupt__" in results:
    print("INTERRUPTED")
    interrupt = results["__interrupt__"][0]
    for request in interrupt.value["action_requests"]:
        print(request["description"])

results = agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config,
)

INTERRUPTED
Tool execution requires approval

Tool: retrieve_content
Args: {'query': '3I/ATLAS 轨道 特征 发现历史'}


In [27]:
messages = results["messages"]
print(f"历史消息：{len(messages)}条")
for message in messages:
    message.pretty_print()

历史消息：6条
================================ Human Message =================================

讲一下3i/Atlas.
================================== Ai Message ==================================

我来帮您查询关于3i/Atlas的信息。
Tool Calls:
  retrieve_content (call_00_JY1oCOy5AwGCGaEgAK8oWg5g)
 Call ID: call_00_JY1oCOy5AwGCGaEgAK8oWg5g
  Args:
    query: 3i/Atlas 彗星 天体
================================= Tool Message =================================
Name: retrieve_content

Source:{'title': '3I ATLAS最新消息 | 3I ATLAS目前位置 | 3I ATLAS外星人, 最新照片 | 3I/亞特拉斯彗星 | Star Walk', 'language': 'zh-Hant', 'description': '2025年12月19日，罕見的星際彗星3I/ATLAS將迎來其與地球的最近距離！這有多危險？了解它對我們星球意味著什麽。\n', 'source': 'https://starwalk.space/zh-Hant/news/3i-atlas-interstellar-object', 'start_index': 1194}
Content:什麽是3I/ATLAS？3I/ATLAS 是已知的第三顆 星際天體——來自太陽系之外的稀有訪客。它于2025年7月1日首次被智利的ATLAS巡天望遠鏡發現。官方觀點（由NASA、ESA以及大多數天文學家支持）很明確：3I/ATLAS是一顆天然彗星——這是繼‘奧陌陌’和彗星2I/鮑裏索夫之後確認的第三個星際天體。但並非所有人都信服，一些人認為它的不尋常特征為更具異域色彩的解釋留下了空間。3I/ATLAS 是外星飛船還是彗星？哈佛教授 vs 科學界自從被發現以來，哈佛天文學家 Avi 

In [28]:
if "__interrupt__" in results:
    print("INTERRUPTED")
    interrupt = results["__interrupt__"][0]
    for request in interrupt.value["action_requests"]:
        print(request["description"])

results = agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config,
)
messages = results["messages"]
print(f"历史消息：{len(messages)}条")
for message in messages:
    message.pretty_print()

INTERRUPTED
Tool execution requires approval

Tool: retrieve_content
Args: {'query': '3I/ATLAS 轨道参数 偏心率 6.23 近日点'}
历史消息：8条
================================ Human Message =================================

讲一下3i/Atlas.
================================== Ai Message ==================================

我来帮您查询关于3i/Atlas的信息。
Tool Calls:
  retrieve_content (call_00_JY1oCOy5AwGCGaEgAK8oWg5g)
 Call ID: call_00_JY1oCOy5AwGCGaEgAK8oWg5g
  Args:
    query: 3i/Atlas 彗星 天体
================================= Tool Message =================================
Name: retrieve_content

Source:{'title': '3I ATLAS最新消息 | 3I ATLAS目前位置 | 3I ATLAS外星人, 最新照片 | 3I/亞特拉斯彗星 | Star Walk', 'language': 'zh-Hant', 'description': '2025年12月19日，罕見的星際彗星3I/ATLAS將迎來其與地球的最近距離！這有多危險？了解它對我們星球意味著什麽。\n', 'source': 'https://starwalk.space/zh-Hant/news/3i-atlas-interstellar-object', 'start_index': 1194}
Content:什麽是3I/ATLAS？3I/ATLAS 是已知的第三顆 星際天體——來自太陽系之外的稀有訪客。它于2025年7月1日首次被智利的ATLAS巡天望遠鏡發現。官方觀點（由NASA、ESA以及大多數天文學家支持）很明確：3I/ATLAS是一顆天然彗星——這是繼